# 3-4 LangGraph 심화

**원본 교안:** [3-4-langgraph-advanced.pdf](./3-4-langgraph-advanced.pdf) · PDF는 그대로 보존합니다.

이 노트북은 슬라이드의 개념 순서를 따라 설명과 확인 실습을 덧붙인 학습본입니다. 로컬 예제는 표준 라이브러리만 사용합니다. 실제 사내 도구·서버 연결은 강의장 환경에서 별도로 확인하세요.


## 학습 목표

- Subgraph, Checkpointer, Store가 해결하는 문제를 구별한다.
- Fan-out/Fan-in과 Send의 병합 조건을 설명한다.


> **실습 전 확인**  기존 실습 폴더에는 이전 회차의 `checkpoints.db`가 남아 있을 수 있습니다. 데모 전 상태를 확인하세요.


## Subgraph · PDF 1–4쪽

[원본 1~4쪽](./3-4-langgraph-advanced.pdf#page=1)

독립적으로 만든 그래프를 부모 그래프의 노드 안에서 호출하면 요약·번역 같은 반복 흐름을 재사용할 수 있습니다. 슬라이드 예시는 자식 그래프에 문서를 전달하고 반환된 요약을 부모 State에 넣습니다. 두 그래프의 State 모양이 다르면 변환 지점을 분명히 해야 합니다.


## Checkpointer와 thread_id · PDF 5–6쪽

[원본 5~6쪽](./3-4-langgraph-advanced.pdf#page=5)

Checkpointer는 실행 중 State를 저장하고 같은 `thread_id`의 흐름을 이어갑니다. `MemorySaver`는 실행 프로세스 안에서만, `SqliteSaver`는 파일에 저장할 수 있습니다. 슬라이드는 `builder.compile(checkpointer=MemorySaver())` 후 같은 `thread_id`로 두 번 호출하는 예를 보여줍니다.


## Memory Store · PDF 7쪽

[원본 7쪽](./3-4-langgraph-advanced.pdf#page=7)

Store는 특정 thread를 넘어 공유할 기억을 보관합니다. `namespace`로 사용자·프로젝트 범위를 나누어 저장·검색합니다. 슬라이드는 `compile(checkpointer=..., store=...)`로 두 저장 장치를 함께 연결합니다. Checkpointer는 대화 흐름, Store는 장기 사실에 가깝습니다.


## 병렬 실행과 Send · PDF 8–10쪽

[원본 8~10쪽](./3-4-langgraph-advanced.pdf#page=8)

Fan-out은 독립 작업을 나누고 Fan-in은 결과를 합칩니다. 정적 분기는 Edge로, 작업 수가 실행 시 정해지는 경우 `Send("worker", state)` 목록으로 표현합니다. 9쪽의 두 갈래(`pros`, `cons`)가 `summary`에서 만나는 그림 같은 코드와 10쪽의 `Send` 목록 생성 예를 비교하세요. 여러 분기가 같은 State 키를 쓴다면 `Annotated` reducer 같은 병합 규칙이 필요합니다.


### 실행해 보기: 상태 병합 생각하기

아래는 라이브러리 없이 병합 규칙의 의미를 확인하는 개념 예제입니다.


In [ ]:
from collections import defaultdict

partial_results = [("장점", "빠름"), ("단점", "근거 확인 필요"), ("장점", "재사용 가능")]
merged = defaultdict(list)
for category, result in partial_results:
    merged[category].append(result)
print(dict(merged))


## CodeMate와 페어 프로그래밍

아래 문장을 CodeMate에 전달하고, 생성된 파일과 실행 결과를 직접 검토하세요. 사내 확장의 메뉴 이름이나 동작은 현장에서 확인합니다.

> Chapter02/03-langgraph-advanced의 네 파일을 읽고 각 예제가 Subgraph, Checkpointer, Store, Send 중 무엇을 보여 주는지 정리해 줘. 03_memory_store.py에서 compile에 무엇이 전달되는지 확인하고, 슬라이드 7쪽과 다르면 동작 차이를 설명해 줘. 변경 전후에 상태가 실제로 저장되는지 확인할 작은 실행 절차를 만들어 줘.


## 결과 검토와 되돌아보기

1. 같은 thread_id를 쓰는 것과 같은 namespace를 쓰는 것은 어떻게 다른가?
2. 병렬 Worker가 같은 키에 값을 쓰면 결과를 어떻게 합쳐야 하는가?
3. 기존 `checkpoints.db`에 대화가 남아 있으면 데모를 어떻게 오해할 수 있는가?
